In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# Frozen step7: actual MP4 save and readback

Click **Run all**. Loads the audited total-step7 POINT4 terminal from run20261001T170749052584Z (true rank1, Delta=5.4846719567056e-7). The fixed primary observation is MP4. No optimization, step6 substitution, new generation or parameter scan.

One native FP32 VAE decode produces a single RGB8 raster for two fixed branches:

- RAW444: materialized RGB8→raw YUV444→reopened RGB24, the previous positive reference channel.
- MP4: the existing project encoder, H264/libx264, CRF18,8fps,yuv420p, followed by independent MP4 readback and native VAE encoding. All181 frames remain320x512; reader stays g0/R44, all174 valid candidates and both keys.

Two normalized observations, four path reads, four payload reads, eight message evaluations,696 valid costs. One VAE decode, two VAE encodes, one MP4 save/read; zero Transformer or gradient calls. This should avoid the previous gradient run's87.7minute workload and disk spool; actual runtime depends on model loading and hardware. No GPU-name gate is added. Existing current-Python dependency repair and fresh-child execution are reused.

The start terminal/result/parent raw444 bytes, normalized observation and both raw readers are checked against their saved identities. Historical OFF/PAYLOAD_MULTI/OVERLAP_MULTI controls remain references, not new FPR samples. Fresh RAW444 replay differences are reported, not hidden. A failed branch remains in the fixed denominator while the other branch can finish.

Inspect both branches' Delta, rank, nearest competition in retained costs, public-margin shortfall, payload and actual RGB quality. Quality is relative to the same marked input raster and raw444, not OFF. RAW444-vs-MP4 changes both chroma subsampling and lossy compression, so a difference does not isolate H264 alone. Exact repeated payload bits remain a separate readout from path recovery.

This is one frozen terminal, one fixed codec configuration, fixed phase and one source. It does not establish generation-time trajectory watermarking, unknown-phase/fragment synchronization, calibrated FPR or broad robustness. Source is immutable when bound below. Local validation is CPU/static with simulated VAE; real pretrained VAE execution is left to the user.


In [ ]:
SOURCE_SHA = '9384030aa3f6f9362daa9deb4126b585996446d7'
from pathlib import Path
import datetime,json,os,subprocess,sys,traceback
STAMP=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUTPUT=Path('/content/drive/MyDrive/Video-WM/Zero-Mean-Step7-MP4-V1')/STAMP
OUTPUT.mkdir(parents=True,exist_ok=False)
REPO=Path('/content/SC-SSTW-STEP7-MP4-'+STAMP)
RUN_OUTPUT=OUTPUT/'fixed_reference'
PYTHON=sys.executable
INPUT_ROOT=Path('/content/drive/MyDrive/Video-WM/Zero-Mean-Channel-Margin-Continue-V1/20261001T170749052584Z/fixed_reference')
REFERENCE_ROOT=Path('/content/drive/MyDrive/Video-WM/Zero-Mean-C1-YUV444-No-H264-V1/20261001T110124499275Z/fixed_reference')
setup=dict(status='SETUP_STARTED',source_sha=SOURCE_SHA,fixed_denominator={'observations': 2, 'mp4_saved': 1, 'mp4_read': 1, 'path_reads': 4, 'payload_reads': 4, 'message_evaluations': 8, 'valid_costs': 696, 'catalog_slots': 15660, 'structural_exclusions': 14964, 'reference_normalized': 3, 'reference_raw': 6, 'parent_normalized': 1, 'parent_raw': 2},observations={p:{'status':'NOT_RUN_SETUP'} for p in ('RAW444','MP4')})
def write_json(path,value):
    temp=path.with_suffix(path.suffix+'.tmp');temp.write_text(json.dumps(value,indent=2)+'\n');os.replace(temp,path)
def failed(stage,exc):
    setup.update(status='SETUP_FAILED',stage=stage,error=f'{type(exc).__name__}: {exc}')
    write_json(OUTPUT/'setup_receipt.json',setup)
    write_json(OUTPUT/'setup_failure.json',dict(stage=stage,error=str(exc),traceback=traceback.format_exc()))
def logged(command,stage,cwd=None,check=True):
    try:
        with (OUTPUT/'execution.log').open('a') as log:
            log.write('COMMAND '+repr(command)+'\n');log.flush()
            child=subprocess.Popen(command,cwd=cwd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
            for line in child.stdout:print(line,end='',flush=True);log.write(line);log.flush()
            code=child.wait();log.write('EXIT '+str(code)+'\n')
        if check and code:raise subprocess.CalledProcessError(code,command)
        return code
    except Exception as exc:failed(stage,exc);raise
write_json(OUTPUT/'setup_receipt.json',setup)
print('Full writer terminal:',INPUT_ROOT,'Saved references:',REFERENCE_ROOT,'Output:',OUTPUT)
if SOURCE_SHA is None:
    error=RuntimeError('Unpublished draft: bind verified published source SHA before Run all')
    failed('SOURCE_BINDING',error);raise error


In [ ]:
try:
    logged(['git','clone','--filter=blob:none','https://github.com/RICHAAARC/SC-SSTW.git',str(REPO)],'SOURCE_CLONE')
    logged(['git','-C',str(REPO),'fetch','origin',SOURCE_SHA],'SOURCE_FETCH')
    logged(['git','-C',str(REPO),'checkout','--detach',SOURCE_SHA],'SOURCE_CHECKOUT')
    actual=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
    dirty=subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True)
    if actual!=SOURCE_SHA or dirty:raise RuntimeError('source SHA/clean checkout mismatch')
    write_json(OUTPUT/'source_receipt.json',dict(source_sha=actual,clean=True))
    torch_ok=logged([PYTHON,'-c','import torch,torchvision; print(torch.__version__,torchvision.__version__)'],'TORCH_PROBE',check=False)==0
    if not torch_ok:
        logged([PYTHON,'-m','pip','install','--upgrade','torch','torchvision'],'TORCH_REPAIR')
    logged([PYTHON,'-m','pip','install','-r',str(REPO/'experiments/wan_state_clock/requirements-grow-video-reference.txt')],'DEPENDENCIES')
    dependency_code=logged([PYTHON,'-m','pip','check'],'DEPENDENCY_REPORT',check=False)
    setup['dependency_check_returncode']=dependency_code
    if dependency_code:print('Metadata conflicts retained; actual imports determine availability.')
    (OUTPUT/'environment_freeze.txt').write_text(subprocess.check_output([PYTHON,'-m','pip','freeze'],text=True))
    import_code='import json,torch; from diffusers import AutoencoderKLWan; from experiments.wan_state_clock.zero_mean_c1_yuv444_no_h264_run import environment; r=environment(); r.update(cuda_available=torch.cuda.is_available()); print(json.dumps(r))'
    env=json.loads(subprocess.check_output([PYTHON,'-c',import_code],cwd=REPO,text=True))
    write_json(OUTPUT/'environment_receipt.json',env)
    if not env['cuda_available']:print('CUDA unavailable: CPU path selected; much slower.')
    logged(['ffmpeg','-version'],'FFMPEG_PROBE');logged(['ffprobe','-version'],'FFPROBE_PROBE')
    setup.update(status='SETUP_COMPLETE',python_executable=PYTHON,environment_strategy='current_python_fresh_children')
    write_json(OUTPUT/'setup_receipt.json',setup)
except Exception as exc:failed('SOURCE_OR_ENVIRONMENT',exc);raise


In [ ]:
try:
    from google.colab import userdata
    token=userdata.get('HF_TOKEN')
    if token:os.environ['HF_TOKEN']=token
except Exception:
    pass
finally:
    token=None
command=[PYTHON,'-u','-m','experiments.wan_state_clock.zero_mean_step7_mp4_run','--output',str(RUN_OUTPUT),'--input-root',str(INPUT_ROOT),'--reference-root',str(REFERENCE_ROOT)]
try:
    returncode=logged(command,'FIXED_RUN',cwd=REPO,check=False)
    RESULT_PATH=RUN_OUTPUT/'result.json'
    write_json(OUTPUT/'execution_receipt.json',dict(command=command,returncode=returncode,result_path=str(RESULT_PATH),result_exists=RESULT_PATH.is_file()))
    if not RESULT_PATH.is_file():raise RuntimeError('No runner result; full setup denominator is retained')
    setup.update(status='SUPERSEDED_BY_RESULT',result_path=str(RESULT_PATH));write_json(OUTPUT/'setup_receipt.json',setup)
except Exception as exc:failed('RUNNER',exc);raise


In [ ]:
result=json.loads(RESULT_PATH.read_text())
print('Status:',result['status'],'Counts:',result.get('counts'),'Calls:',result['calls'])
print('Decoded raster matches parent:',result.get('decoded_raster_matches_parent'))
print('RAW444 baseline replay:',result.get('baseline_replay'))
for name,row in result['comparisons'].items():print(name,row)
print('Fixed endpoint:',result.get('fixed_endpoint'))
print('MP4:',result['mp4'])
for name,row in result['posthoc'].items():print(name,row)
for name,row in result['message_evaluations'].items():print(name,row)
for name,row in result['observations'].items():
    print(name,'status=',row['status'],'quality_vs_shared_q8=',row.get('quality_vs_shared_q8'),
          'quality_vs_parent_raw444=',row.get('quality_vs_parent_raw444'),
          'quality_vs_current_raw444=',row.get('quality_vs_current_raw444'))
print('Failures:',result['failures'])
print(result['evidence_ceiling'])
print('Retained result:',RESULT_PATH)
